# Rung 21 — one training run, ONE recipe flag, three per-epoch readouts

**Baseline / control:** rung 18 (`18_count_aug_v1`), all three epochs already scored —
`bucket_mean` 0.5255 / 0.5488 / **0.5721**, Spearman r on `Clips` 0.4888 / 0.5610 / **0.6003**.

**This rung changes no data.** It trains on rung 18's own `train.jsonl`, in place, sha256
asserted. The only thing that moves is one recipe flag:

| arm | flag | control → arm |
|---|---|---|
| `A_lr` | `--learning_rate` | 2e-5 → **1e-4** |
| `B_rank` | `--lora_rank` + `--lora_alpha` | 8/32 → **32/128** (α/r held at 4) |

**Why the recipe and why now** — nineteen rungs changed data, prompt, input or output and
none ever touched the optimiser. Every strong surgical-VQA result pairs lr 1e-5–2e-5 with
15–20 epochs, or 3–6 epochs with lr 1e-4–3e-4; we run lr 2e-5 for 3 epochs, the only cell in
the published grid taking both discounts. See `PLAN.md` and
`context/decisions/undertrained-on-both-axes.md`.

⚠️ **Read the PLAN's named risk before interpreting a collapse.** Our LoRA reaches the ViT at
the LLM's own learning rate; the Qwen3-VL default puts the tower 5–10× lower. At 1e-4 the
tower gets 1e-4 too, so a collapse in arm A may be the tower rather than the recipe. The
pre-registered diagnostic is a follow-up arm (`vit_lr` held at 2e-5), not an edit to this one.


In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, re, sys, time
from dataclasses import replace
from pathlib import Path

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent

# `swift` is shelled out to by _train / merge. A papermill kernel does NOT inherit the env's
# bin/ on PATH, and it must be THIS interpreter's bin so the CLI and the kernel come from one
# environment.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models", EXP / "_tools",
          REPO / "experiments" / "18-count-aug" / "_models",   # the control's engine
          REPO / "experiments" / "06-vit-lora" / "_models",    # the recipe itself
          REPO / "experiments" / "02-lora-sft" / "_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

# The SDK judge / any HF lookup is cached at /workspace/hf_cache on this pod, not at the
# default ~/.cache/huggingface. Set it BEFORE the offline flags mean anything.
os.environ.setdefault("HF_HOME", "/workspace/hf_cache")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

import torch
print("torch", torch.__version__, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "->", _envbin)
print("repo:", REPO)


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects its overrides BELOW this cell) ----
SMOKE = True          # full run: -p SMOKE False
SEED  = 42

# 🎯 THE ARM. One of: "control", "A_lr", "B_rank".  Full run: -p ARM A_lr
ARM = "A_lr"

# The recipe values each arm asks for. The CONTROL's values are 2e-5 / 8 / 32 — they are
# rung 18's, and `assert_control_is_rung18` reads them off rung 18's own config object rather
# than trusting these literals.
LR_A       = 1e-4     # arm A_lr   — Qwen3-VL community default; bottom of the 1e-4..3e-4 band
RANK_B     = 32       # arm B_rank — IOVQA measures rank 32->128 = +0.028 at our data scale
ALPHA_B    = 128      # ⚠️ moves WITH rank: alpha/rank must stay 4 or the arm is an LR change

EPOCHS      = 3       # epoch-matched to rung 18. Epochs 4+ would have no control (RULES 6b)
PER_DEVICE  = 1       # MEASURED last session: 4x4 and 6x2 OOM on this 32 GB card, and
GRAD_ACCUM  = 16      # 1x16 is 12% faster and 4.2 GB lighter than 2x8. Product must stay 16.

SMOKE_STEPS = 4
PROBE_STEPS = 6       # >2: rung 06's first smoke read 110 s/it and that was all warm-up
VRAM_PROBE  = True    # arm B adds optimiser state; rung 18 peaked 22,210 of 32,607 MiB

RUN_A     = "21_lr_1e4_v1"
RUN_B     = "21_rank32_v1"
RUN_CTL   = "21_control_v1"
SMOKE_RUN = "21_smoke"

MODEL_BASE    = "/workspace/models/qwen3-vl-8b"
CONTROL_RUN   = "/workspace/repo/experiments/18-count-aug/runs/18_count_aug_v1"
CONTROL_SHA   = ""    # "" = record it on the first pass, then paste it here and it is asserted


In [ ]:
# --- derived (MUST live BELOW the parameters cell) ------------------------------
# 🔴 papermill injects its override cell immediately AFTER the cell tagged `parameters`. A
# value DERIVED inside that cell is computed from the pre-injection literals and is never
# recomputed, so `-p SMOKE False` would change SMOKE and nothing else. Rung 16's first "full"
# run was the smoke again (n=64) and was indistinguishable from a success. Every derived value
# lives here, below the injection point, and G0 asserts the REALIZED artifact matches the
# declared mode.
from recipe_sweep_train import (
    ARMS, RecipeSweepConfig, assert_control_is_rung18, assert_dataset_is_the_controls,
    assert_single_variable, control_cfg, diff_vs_control, effective_batch, list_checkpoints,
    main, merge_checkpoint, read_g1, train_with_vram,
)

_RECIPE = {
    "control": dict(learning_rate=2e-5, lora_rank=8,     lora_alpha=32,      run=RUN_CTL),
    "A_lr":    dict(learning_rate=LR_A, lora_rank=8,     lora_alpha=32,      run=RUN_A),
    "B_rank":  dict(learning_rate=2e-5, lora_rank=RANK_B, lora_alpha=ALPHA_B, run=RUN_B),
}
if ARM not in _RECIPE:
    raise ValueError(f"ARM={ARM!r} is not one of {sorted(_RECIPE)}")
_arm = _RECIPE[ARM]

RUN_NAME = SMOKE_RUN if SMOKE else _arm["run"]
CONTROL  = Path(CONTROL_RUN)

cfg = RecipeSweepConfig(
    exp_dir=EXP,
    run_name=RUN_NAME,
    model_path=Path(MODEL_BASE),
    learning_rate=_arm["learning_rate"],
    lora_rank=_arm["lora_rank"],
    lora_alpha=_arm["lora_alpha"],
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE,
    gradient_accumulation_steps=GRAD_ACCUM,
    seed=SEED,
    smoke=SMOKE,
    smoke_max_steps=SMOKE_STEPS,
    control_train_jsonl=CONTROL / "train.jsonl",
    control_sha256=CONTROL_SHA or None,
    # eval_loss is observability ONLY and never selects a checkpoint — the control's own
    # val.jsonl, so not even the observability channel differs.
    val_jsonl=(CONTROL / "val.jsonl") if (CONTROL / "val.jsonl").exists() else None,
    run_guard=not SMOKE,
)
cfg.run_dir.mkdir(parents=True, exist_ok=True)

print(f"ARM        {ARM}")
print(f"SMOKE      {SMOKE}   (steps={SMOKE_STEPS})")
print(f"run dir    {cfg.run_dir}")
print(f"recipe     lr={cfg.learning_rate}  r={cfg.lora_rank}  alpha={cfg.lora_alpha}  "
      f"(alpha/r={cfg.lora_alpha / cfg.lora_rank})  epochs={cfg.num_train_epochs}")
print(f"batch      {cfg.per_device_train_batch_size} x {cfg.gradient_accumulation_steps} "
      f"= {effective_batch(cfg)}")
print(f"dataset    {cfg.train_jsonl}   (the control's file, in place)")
print(f"guard      {cfg.run_guard}")


## Gate order

Everything cheap runs first, and nothing loads a model until all of it has passed. The gates
RAISE — a gate that fires is a finding, not an obstacle (RULES §7).

| | gate | what it refuses to let through |
|---|---|---|
| G-recipe | `assert_control_is_rung18` | a control that quietly stopped being rung 18 |
| G-var | `assert_single_variable` | a second flag; and α/r moving, which would make arm B a disguised LR change |
| G-data | `assert_dataset_is_the_controls` | training on a re-export instead of the control's bytes |
| G0 | declared-vs-realized | a "full" run that is the smoke again (rung 16's lost day) |
| G-vram | VRAM probe | an OOM three hours into a nine-hour run |
| G1 | `read_g1` | a LoRA that never reached the ViT (`tuner.py:93`'s silent early return) |


In [ ]:
# --- G-recipe + G-var: the control is rung 18, and exactly ONE thing differs -----
# Both argvs are built by rung 06's own `_swift_args`, never hand-typed: if the recipe
# function ever changes, both sides change together and the diff stays honest.
ctl_check = assert_control_is_rung18(cfg)
var_check = assert_single_variable(cfg, ARM)

print("control == rung 18 on:", ", ".join(ctl_check["checked_fields"]))
print(f"\narm {ARM!r} diff vs control (must be exactly {sorted(ARMS[ARM]) or 'nothing'}):")
for flag, (was, now) in sorted(var_check["diff"].items()):
    print(f"   {flag:<24} {was}  ->  {now}")
print(f"\nalpha/rank {var_check['alpha_over_rank']}   effective batch "
      f"{var_check['effective_batch']}")


In [ ]:
# --- G-data: train on the CONTROL's bytes, not on a copy that looks the same ----
data_check = assert_dataset_is_the_controls(cfg)
print(json.dumps(data_check, indent=2))

# The arithmetic that makes the cost checkable against a file: 14,415 / 16 = 900.9 steps per
# epoch, and 3 epochs = 2,703 steps — which is rung 18's own `checkpoint-2703`.
_spe = data_check["steps_per_epoch"]
print(f"\nprojected: {_spe:.1f} steps/epoch  ->  {_spe * cfg.num_train_epochs:.0f} steps "
      f"for {cfg.num_train_epochs} epochs  ->  "
      f"{_spe * cfg.num_train_epochs * 11.56 / 3600:.2f} h at rung 18's 11.56 s/it")

if not SMOKE and not data_check["sha256"] == (CONTROL_SHA or data_check["sha256"]):
    raise AssertionError("unreachable — assert_dataset_is_the_controls already raised")
if not CONTROL_SHA:
    print("\n⚠️  CONTROL_SHA is empty: this pass RECORDS the digest instead of asserting it. "
          "Paste it into the parameters cell before the full run.")


In [ ]:
# --- G0: the REALIZED artifact must match the DECLARED mode ---------------------
# 🔴 Not "is the SMOKE variable False" — that is exactly what rung 16 checked while running
# the smoke for a second time. This reads the argv that will actually be executed.
from vit_lora_train import _swift_args

argv = _swift_args(cfg)
has_max_steps = "--max_steps" in argv
declared_full = not SMOKE

if declared_full and has_max_steps:
    raise AssertionError(
        "declared FULL but the command carries --max_steps — this would run the smoke again "
        "and be indistinguishable from a success"
    )
if not declared_full and not has_max_steps:
    raise AssertionError("declared SMOKE but the command has no --max_steps")
from recipe_sweep_train import _as_map

if declared_full and _as_map(argv).get("--num_train_epochs") != str(cfg.num_train_epochs):
    raise AssertionError(
        f"the argv trains {_as_map(argv).get('--num_train_epochs')} epochs, the config says "
        f"{cfg.num_train_epochs} — epoch-matching to rung 18 is what makes the delta readable"
    )
if declared_full and cfg.run_name == SMOKE_RUN:
    raise AssertionError("declared FULL but writing into the smoke run dir")

print(f"G0 ok — mode={'FULL' if declared_full else 'SMOKE'}, "
      f"run_dir={cfg.run_dir.name}, --max_steps present={has_max_steps}")
print("\ncommand:\n ", " ".join(argv))


In [ ]:
# --- G-vram: has THIS arm been shown to fit the card? ---------------------------
# There is no separate VRAM probe in this rung. The batch shape is settled and frozen, so the
# only open question is whether the ARM fits — and the SMOKE pass runs the real command on the
# real data, which answers it for free and from the exact configuration being committed to.
# Arm B is the one at risk: r=32 quadruples the trainable count, and it lands on the optimiser
# state. Rung 18 peaked 22,210 MiB of 32,607.
VRAM_LOG = EXP / f"RESULTS_vram_{ARM}.json"

if not SMOKE:
    if not VRAM_LOG.exists():
        raise AssertionError(
            f"{VRAM_LOG.name} is missing — run this notebook in SMOKE for this arm first. "
            "An OOM three hours into a nine-hour run costs the whole run."
        )
    smoke_vram = json.loads(VRAM_LOG.read_text())
    if not smoke_vram["ok"]:
        raise AssertionError(f"the smoke for arm {ARM} did not complete: {smoke_vram['error']}")
    if smoke_vram["peak_mib"] > 30_000:
        raise AssertionError(
            f"the smoke peaked {smoke_vram['peak_mib']} MiB of 32,607 — no headroom for a "
            f"{cfg.num_train_epochs}-epoch run. Do NOT lower max_pixels or the batch to make "
            "it fit; each is a second variable."
        )
    print(f"G-vram ok — smoke peaked {smoke_vram['peak_mib']} MiB, "
          f"{smoke_vram['s_per_it']} s/it")
else:
    print("SMOKE pass — this run RECORDS the VRAM/speed the full run will be gated on")


In [ ]:
# --- train ----------------------------------------------------------------------
# STOP RULES (unchanged from rungs 06 and 18): OOM -> STOP; do NOT lower max_pixels / LR /
# effective batch, each is a second variable. Loss diverges -> report it, do not touch the LR.
# ⚠️ Divergence is a PLAUSIBLE outcome of arm A (5x the learning rate) and it is a RESULT.
# The broken-run guard is ON for the full run and OFF in SMOKE: a pure stdout observer, so
# when it does not abort the run is byte-identical to an unguarded one.
t0 = time.perf_counter()
vram = train_with_vram(cfg)
print(f"\n{'smoke' if SMOKE else 'training'} finished in "
      f"{(time.perf_counter() - t0) / 3600:.2f} h")
print(json.dumps(vram, indent=2))

if SMOKE:
    # The full run's G-vram gate reads this file. Written even on failure: an OOM is the
    # single most useful thing a smoke can tell us, and it must not be lost.
    (EXP / f"RESULTS_vram_{ARM}.json").write_text(json.dumps(vram, indent=2))
    if vram["ok"] and vram["s_per_it"]:
        _steps = data_check["steps_per_epoch"] * cfg.num_train_epochs
        print(f"\nprojected full run: {_steps:.0f} steps x {vram['s_per_it']:.2f} s/it = "
              f"{_steps * vram['s_per_it'] / 3600:.2f} h")

if not vram["ok"]:
    raise RuntimeError(vram["error"])
print("checkpoints:", [c.name for c in list_checkpoints(cfg)])


In [ ]:
# --- G1: did the LoRA actually reach the ViT, and how big did the arm get? ------
g1 = read_g1(cfg)
print(json.dumps(g1, indent=2))

if not g1["targets_vision_tower"]:
    raise AssertionError(
        "no vision_tower in target_modules — the LoRA never reached the ViT and this arm is "
        "not comparable to rung 18 (tuner.py:93 returns early on a STRING target_modules)"
    )
if g1["trainable_params_M"] is None or g1["trainable_params_M"] >= 500:
    raise AssertionError(f"trainable params {g1['trainable_params_M']}M — expected < 500M")

(cfg.run_dir / "g1.json").write_text(json.dumps(
    {"arm": ARM, **g1, **var_check, "data": data_check}, indent=2, default=str))
print(f"\nG1 ok — {g1['trainable_params_M']}M trainable, LoRA reaches the ViT")


## What happens next — and what does NOT happen here

This notebook produces checkpoints. **It does not score them.** Merging and evaluating every
epoch is `21b_epoch_eval.ipynb`, for the reason rung 18 learned the hard way: the judge lives
under `HF_HOME=/workspace/hf_cache`, and getting that wrong crashes *after* a 17 GB merge and
a full inference pass.

Read, per epoch, epoch-matched against rung 18's own series (RULES §6b):

1. **The leaderboard proxy — mean(`aggregation_ID`, `object_recognition_ID`)**, with its two
   components beside it. This is what the platform scores (RULES §4b).
2. **`margin_OOD`** — pre-registered as a NO-FALL condition. OOD is 50% of the final ranking.
3. `bucket_mean` vs 0.5255 / 0.5488 / 0.5721.
4. **Spearman r on `Clips`** vs 0.4888 / 0.5610 / 0.6003, quoting template and n (RULES §13b).
   A rise in `r` is a rise in `r` — it does not convert into points (§13c).
5. 🆕 **class-balanced F1 on `fo_class`** (`frame.metrics.class_f1_report`). More optimisation
   distance is most likely to do its damage in the tail, and the tail is what the 0.6478
   headline cannot see: rung 18 ep3 reads `Gallstone` recall **0.036**.
6. `eval_loss` per epoch — observability only, never selection. Rung 18's *rose* at epoch 3
   while its scored `bucket_mean` improved.

⚠️ **No variance estimate exists in this project.** Quote the video-clustered paired bootstrap
CI (`frame.metrics.paired_delta_ci`) against rung 18's answers, or do not quote the delta.
